# VGG16 Chest X-ray Classification (NORMAL vs PNEUMONIA)

Transfer learning with a frozen VGG16 feature extractor and a new 2-class classifier head.

Expected folder layout relative to the notebook:

```
data/
    train/  NORMAL/ PNEUMONIA/
    val/    NORMAL/ PNEUMONIA/
    test/   NORMAL/ PNEUMONIA/
models/     (created automatically)
```

## 1. Imports

In [1]:
import random
import shutil
from pathlib import Path

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

## 2. Configuration

In [2]:
PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "full_data"
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "vgg16_best.pth"

# Split settings
SPLIT_RATIOS = (0.70, 0.15, 0.15)  # train, val, test
SPLIT_SEED = 42

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
NUM_CLASSES = 2  # 0 -> NORMAL, 1 -> PNEUMONIA

EPOCHS = 5
LEARNING_RATE = 0.0001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## 3. SPLIT RAW DATA INTO TRAIN / VAL / TEST

In [3]:
def split_dataset(raw_dir, data_dir, ratios=(0.70, 0.15, 0.15), seed=42):
    if not raw_dir.is_dir():
        raise FileNotFoundError(f"Raw data directory not found: {raw_dir}")

    if TRAIN_DIR.is_dir() and any(TRAIN_DIR.iterdir()):
        print("Split already exists, skipping.")
        return

    if abs(sum(ratios) - 1.0) > 1e-6:
        raise ValueError("Split ratios must sum to 1.0")

    rng = random.Random(seed)
    class_dirs = sorted(p for p in raw_dir.iterdir() if p.is_dir())

    if not class_dirs:
        raise FileNotFoundError(f"No class folders found in {raw_dir}")

    for class_dir in class_dirs:
        files = sorted(f for f in class_dir.iterdir() if f.is_file())
        rng.shuffle(files)

        n_total = len(files)
        n_train = int(n_total * ratios[0])
        n_val = int(n_total * ratios[1])

        splits = {
            "train": files[:n_train],
            "val": files[n_train:n_train + n_val],
            "test": files[n_train + n_val:],
        }

        for split_name, split_files in splits.items():
            out_dir = data_dir / split_name / class_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            for f in split_files:
                shutil.copy2(f, out_dir / f.name)

        print(
            f"{class_dir.name}: {n_total} images -> "
            f"train {len(splits['train'])}, "
            f"val {len(splits['val'])}, "
            f"test {len(splits['test'])}"
        )

    print("Split complete.")


split_dataset(RAW_DIR, DATA_DIR, ratios=SPLIT_RATIOS, seed=SPLIT_SEED)


NORMAL: 1583 images -> train 1108, val 237, test 238
PNEUMONIA: 4273 images -> train 2991, val 640, test 642
Split complete.


## 4. Dataset: preprocessing, loading and dataloaders

In [4]:
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])


def load_datasets():
    for directory in (TRAIN_DIR, VAL_DIR, TEST_DIR):
        if not directory.is_dir():
            raise FileNotFoundError(
                f"Dataset directory not found: {directory}"
            )

    train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=image_transform)
    val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=image_transform)
    test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=image_transform)

    expected_classes = train_dataset.class_to_idx

    if val_dataset.class_to_idx != expected_classes:
        raise ValueError("Validation classes do not match training classes.")

    if test_dataset.class_to_idx != expected_classes:
        raise ValueError("Test classes do not match training classes.")

    return train_dataset, val_dataset, test_dataset


train_dataset, val_dataset, test_dataset = load_datasets()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

Classes: ['NORMAL', 'PNEUMONIA']
Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}
Training images: 4099
Validation images: 877
Test images: 880


## 5. Model: VGG16 with a new classification head

In [11]:
def build_model(pretrained=True):
    """
    Create a VGG16 model for binary chest X-ray classification.

    Classes:
        0 -> NORMAL
        1 -> PNEUMONIA
    """
    weights = models.VGG16_Weights.DEFAULT if pretrained else None

    model = models.vgg16(weights=weights)

    # Freeze the convolutional feature extractor initially.
    for parameter in model.features.parameters():
        parameter.requires_grad = False

    # Replace the original ImageNet classification layer.
    input_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(
        in_features=input_features,
        out_features=NUM_CLASSES,
    )

    return model.to(DEVICE)

## 6. Training and validation functions

In [5]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    # Keep the frozen feature extractor in evaluation mode.
    model.features.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_images += batch_size

    average_loss = total_loss / total_images
    accuracy = correct_predictions / total_images

    return average_loss, accuracy


def validate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)
            correct_predictions += (predictions == labels).sum().item()
            total_images += batch_size

    average_loss = total_loss / total_images
    accuracy = correct_predictions / total_images

    return average_loss, accuracy

## 6. Main training process

Trains for `EPOCHS` epochs and saves the checkpoint whenever validation loss improves.

In [9]:
def train_model():
    print("Starting VGG16 training...", flush=True)
    print(f"Device: {DEVICE}", flush=True)
    print(f"Training images: {len(train_dataset)}", flush=True)
    print(f"Validation images: {len(val_dataset)}", flush=True)

    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    model = build_model(pretrained=True)

    criterion = nn.CrossEntropyLoss()

    # Only optimize parameters that are not frozen.
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
    )

    best_val_loss = float("inf")

    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch + 1}/{EPOCHS}", flush=True)

        train_loss, train_accuracy = train_one_epoch(
            model, train_loader, criterion, optimizer
        )
        val_loss, val_accuracy = validate(model, val_loader, criterion)

        print(
            f"Train loss: {train_loss:.4f} | "
            f"Train accuracy: {train_accuracy:.4f}",
            flush=True,
        )
        print(
            f"Validation loss: {val_loss:.4f} | "
            f"Validation accuracy: {val_accuracy:.4f}",
            flush=True,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                    "val_accuracy": val_accuracy,
                    "class_to_idx": train_dataset.class_to_idx,
                },
                MODEL_PATH,
            )

            print(f"Saved best checkpoint: {MODEL_PATH}", flush=True)

    print("\nTraining finished.", flush=True)
    print(f"Best validation loss: {best_val_loss:.4f}", flush=True)

    return model

## 7. Run training

In [12]:
model = train_model()

Starting VGG16 training...
Device: cuda
Training images: 4099
Validation images: 877
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/mega-pc/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [01:20<00:00, 6.85MB/s] 



Epoch 1/5
Train loss: 0.1684 | Train accuracy: 0.9329
Validation loss: 0.1006 | Validation accuracy: 0.9647
Saved best checkpoint: /home/mega-pc/pneumo-xai/models/vgg16_best.pth

Epoch 2/5
Train loss: 0.1041 | Train accuracy: 0.9656
Validation loss: 0.1059 | Validation accuracy: 0.9692

Epoch 3/5
Train loss: 0.0763 | Train accuracy: 0.9722
Validation loss: 0.0915 | Validation accuracy: 0.9761
Saved best checkpoint: /home/mega-pc/pneumo-xai/models/vgg16_best.pth

Epoch 4/5
Train loss: 0.0575 | Train accuracy: 0.9807
Validation loss: 0.1059 | Validation accuracy: 0.9658

Epoch 5/5
Train loss: 0.0509 | Train accuracy: 0.9810
Validation loss: 0.1384 | Validation accuracy: 0.9692

Training finished.
Best validation loss: 0.0915


## 8. Sanity check: model output shape

Quick check that the model accepts a batch of 224x224 RGB images and returns 2 logits per image.

In [13]:
print("Initializing VGG16 for shape check...", flush=True)
check_model = build_model(pretrained=True)

sample_images = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
check_model.eval()

with torch.no_grad():
    predictions = check_model(sample_images)

print("Input shape:", sample_images.shape)
print("Output shape:", predictions.shape)
print("Device:", next(check_model.parameters()).device)

Initializing VGG16 for shape check...
Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 2])
Device: cuda:0
